# Method 2 — Cross-Encoder (Kaggle T4)

Phase 3 của `docs/method2_plan.md`: sinh nhãn, curriculum 2 giai đoạn, gate theo head. Ngân sách ~4.5h GPU.

**Ba quy tắc sống còn trên Kaggle** (§8 `docs/method2_plan.md`):

1. Bật **Save & Run All (Commit)** cho job dài — session tương tác bị ngắt sau ~20 phút không tương tác, commit run chạy nền đủ 12h.
2. Checkpoint mỗi 500 step vào `/kaggle/working`, và **luôn** hỗ trợ `resume_from`.
3. Cache model HuggingFace thành Kaggle Dataset (`BAAI/bge-m3` ~2.3GB) thay vì tải lại mỗi session.


In [ ]:
# ===== Cell 0: dò dataset + HF cache =====
# PHẢI chạy trước mọi import transformers: thư viện chốt cache lúc import,
# set HF_HOME sau đó thì không còn tác dụng.
import os
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')


def _dirs_within(base: Path, max_depth: int = 4):
    """Mọi thư mục tới độ sâu `max_depth`, bỏ qua `hub/` cho nhanh."""
    frontier, seen = [base], []
    for _ in range(max_depth):
        nxt = []
        for d in frontier:
            try:
                children = [c for c in d.iterdir() if c.is_dir() and c.name != 'hub']
            except (PermissionError, OSError):
                continue
            seen.extend(children)
            nxt.extend(children)
        frontier = nxt
    return seen


def find_root(marker: str, label: str) -> Path:
    """Tìm thư mục chứa `marker`.

    Kaggle mount theo dạng /kaggle/input/datasets/<user>/<ds>/<ds>/, và số tầng
    đổi theo cách upload. Dò theo marker thì không phải hardcode username hay
    độ sâu — upload kiểu nào cũng tìm ra.
    """
    for d in [INPUT_ROOT] + _dirs_within(INPUT_ROOT):
        if (d / marker).exists():
            return d
    raise SystemExit(
        f'Không tìm thấy {label}: không thư mục nào dưới {INPUT_ROOT} có {marker}.\n'
        'Kiểm tra đã Add đủ 3 dataset ở sidebar Input chưa.'
    )


SRC_ROOT = find_root('src/models/preflight.py', 'dataset src')
DATA_ROOT = find_root('method2/manifest.json', 'dataset data')
HF_HOME = find_root('hub/models--BAAI--bge-m3', 'dataset hf-cache')

print('SRC :', SRC_ROOT)
print('DATA:', DATA_ROOT)
print('HF  :', HF_HOME)

os.environ['HF_HOME'] = str(HF_HOME)
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

# Có thư mục model chưa đủ — thiếu file trọng số thì lỗi chỉ lộ ra lúc nạp
# model, sau khi đã tốn thời gian cài đặt và copy.
for name in ('models--BAAI--bge-m3', 'models--xlm-roberta-base'):
    weights = [
        f for f in (HF_HOME / 'hub' / name).rglob('*')
        if f.is_file() and f.suffix in ('.safetensors', '.bin') and f.stat().st_size > 10**8
    ]
    assert weights, f'{name}: không có file trọng số > 100 MB'
    print(f'  {name}: {max(f.stat().st_size for f in weights) / 1024**3:.2f} GB')
print('\nHF cache OK')


In [ ]:
# ===== Cell 1: env — PIN version =====
# Ba package này quyết định API training VÀ tên metric của
# InformationRetrievalEvaluator. Đổi bản là đổi khoá metric, hỏng cả
# load_best_model_at_end lẫn khả năng so sánh giữa các run.
!pip install -q 'transformers==5.15.1' 'sentence-transformers==6.0.0' 'peft==0.20.0' \
                accelerate jsonschema rank_bm25 datasets

# torch KHÔNG pin: Kaggle cài sẵn bản CUDA riêng, ép cài lại vừa chậm vừa
# dễ lệch CUDA runtime của image. Chỉ ghi nhận version vào manifest.
import torch

free, total = torch.cuda.mem_get_info()
print(torch.cuda.get_device_name(0), f'{free/1024**3:.1f} / {total/1024**3:.1f} GB free')
print('bf16 supported:', torch.cuda.is_bf16_supported())
# Kỳ vọng: Tesla T4, ~15.0 GB free, bf16 = False → mọi config dùng fp16.


In [ ]:
# ===== Cell 2: copy code + data vào /kaggle/working =====
# Dataset chỉ đọc, mà code ghi checkpoint và dùng đường dẫn tương đối, nên
# phải copy sang thư mục ghi được. Dùng path đã dò ở Cell 0.
import shutil

WORK = Path('/kaggle/working')
for name in ('src', 'configs'):
    target = WORK / name
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(SRC_ROOT / name, target)

# Dataset data bắt đầu thẳng bằng method2/ custom_vi/ benchmark_vi/ (KHÔNG có
# tầng `data/`), còn code tham chiếu `data/method2/...` → copy vào data/.
data_dir = WORK / 'data'
if data_dir.exists():
    shutil.rmtree(data_dir)
data_dir.mkdir(parents=True)
for child in DATA_ROOT.iterdir():
    dest = data_dir / child.name
    shutil.copytree(child, dest) if child.is_dir() else shutil.copy2(child, dest)

%cd /kaggle/working

import json, glob, sys
sys.path.insert(0, '/kaggle/working')
# HF_HOME đã set ở Cell 0, kế thừa sang mọi tiến trình con `!python`.

print('src    :', sorted(p.name for p in (WORK / 'src').iterdir()))
print('data   :', sorted(p.name for p in data_dir.iterdir()))


In [ ]:
# ===== Cell 3: kiểm tra bản copy TRƯỚC khi preflight =====
# Preflight kiểm tra tính đúng đắn của dữ liệu; cell này kiểm tra bước copy —
# tách ra để khi hỏng thì biết ngay là hỏng ở đâu.
REQUIRED = [
    'data/method2/decontamination.json',
    'data/method2/manifest.json',
    'data/method2/tool_pool.json',
    'data/method2/biencoder/train.jsonl',
    'data/method2/biencoder/val.jsonl',
    'data/method2/biencoder/pairs_stats.json',
    'data/method2/crossencoder/train.jsonl',
    'data/method2/crossencoder/val.jsonl',
    'data/method2/label_stats.json',
    'data/custom_vi/v1/test_seen.jsonl',
    'data/benchmark_vi/test.jsonl',
    'configs/method2/biencoder.yaml',
    'configs/method2/pinned_versions.json',
    'src/models/preflight.py',
]
missing = []
for rel in REQUIRED:
    path = WORK / rel
    if path.exists() and path.stat().st_size > 0:
        print(f'  {path.stat().st_size / 1024**2:8.2f} MB  {rel}')
    else:
        missing.append(rel)
        print(f'  {"THIẾU":>11}  {rel}')
assert not missing, f'Copy chưa đủ: {missing}'

# import được thì mới chắc src/ copy nguyên vẹn.
import importlib

importlib.import_module('src.models.preflight')
manifest = json.load(open('data/method2/manifest.json', encoding='utf-8'))
print('\nsnapshot commit:', manifest.get('git_commit'))
print('copy OK')


## Sinh cặp (query, parameter) và kiểm tra tỉ lệ SKIP

**Gate §1.5**: nếu %SKIP > 30% thì dừng lại, xem lại quyết định Q2 (có thêm
fuzzy span alignment hay không) trước khi train.


In [ ]:
!python -m src.models.crossencoder.dataset --config configs/method2/crossencoder.yaml

stats = json.load(open('data/method2/label_stats.json', encoding='utf-8'))
print('SKIP rate:', stats['skip_rate'])
print('theo lý do:', stats['skip_by_reason'])
print('unsupported coverage:', stats['unsupported_type_coverage'])
assert stats['skip_rate'] <= 0.30, 'SKIP quá ngưỡng — xem lại Q2 trước khi train'


## Train — curriculum 2 giai đoạn, resume-safe

1. Warm-up trên glaive+xLAM (2 epoch) — học kỹ năng span tổng quát.
2. Fine-tune trên custom_vi (2 epoch, lr 1e-5) — domain đích.


In [ ]:
RUN = '/kaggle/working/artifacts/method2/crossencoder/run01'
resume = sorted(glob.glob(f'{RUN}/checkpoint-*'))[-1] if glob.glob(f'{RUN}/checkpoint-*') else None
print('resume from:', resume)

!python -m src.models.crossencoder.train \
    --config configs/method2/crossencoder.yaml \
    --output-dir {RUN} \
    --resume-from {resume}


## Gate để qua Phase 4 (chế độ oracle retrieval)

`has_value` F1 ≥ 0.90 · Span EM ≥ 0.80 · Enum acc ≥ 0.90 · Boolean acc ≥ 0.85


In [ ]:
!python -m src.models.crossencoder.evaluate \
    --config configs/method2/crossencoder.yaml \
    --model {RUN}/final \
    --pairs data/method2/crossencoder/val.jsonl \
    --output results/method2/metrics/crossencoder_val.json


In [ ]:
# ===== Lưu artifact =====
# Kaggle chỉ giữ /kaggle/working (20GB). Nén để tải về hoặc làm Dataset mới.
!tar czf /kaggle/working/crossencoder_run.tar.gz -C /kaggle/working/artifacts/method2 .
!du -h /kaggle/working/crossencoder_run.tar.gz
